In [1]:
!pip install pylabwons

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.7 MB/s eta 0:00:00a 0:00:01
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=bc21d7ee9215ae088328e2b1ae06f2bd8caf59e0c031bdec959c3bbedf650322
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


# ASSET

| 구분 | XLE (대형주 중심) | XOP (탐사/생산 중심) | WTIU (3배 레버리지) |
| :--- | :--- | :--- | :--- |
| **기초 자산** | S&P 500 에너지 섹터 기업 | S&P Oil & Gas E&P 지수 | Solactive MicroSectors US Big Oil Index |
| **가중 방식** | **시가총액 가중** (대형주 비중↑) | **동일 가중** (모든 종목 균등 비중) | **동일 가중** (상위 12개 대형주 중심) |
| **운용 방식** | 1배수 (현물 ETF) | 1배수 (현물 ETF) | **일일 수익률 3배 추종 (ETN)** |
| **주요 종목** | 엑손모빌, 쉐브론 등 대기업 | 중소형 탐사·생산(E&P) 기업 | 엑손모빌, 쉐브론 등 우량주 12개 |
| **운용 보수** | **0.09%** | 0.35% | **0.95%** |
| **주요 특징** | 안정적인 대형주 및 배당 중심 | 유가 변동에 민감한 중소형주 포함 | 단기 방향성 매매용 (변동성 매우 높음) |

In [2]:
import yfinance as yf
import pandas as pd
import pandas_datareader.data as web
from datetime import datetime


# XOP: Energy Select Sector SPDR Fund
# XLE: SPDR S&P Oil & Gas Exploration & Production ETF
xop = yf.Ticker('XOP')
xle = yf.Ticker('XLE')
lev = yf.Ticker('WTIU')

objs = {}
for ticker in [xop, xle, lev]:
    ohlcv = ticker.history(period='10y', interval='1d')[['Open', 'High', 'Low', 'Close', 'Volume']]
    ohlcv.columns = ['open', 'high', 'low', 'close', 'volume']
    objs[ticker.ticker] = ohlcv
    

asset = pd.concat(objs, axis=1)
asset = asset.tz_localize(None) if asset.index.tz is not None else asset
# asset

# ANALYTICS

## 01. PE

In [5]:
print(f'XLE: \n  - TRAILING PE: {xle.info.get("trailingPE")}\n  - FORWARD PE: {xle.info.get("forwardPE")}')
print(f'XOP: \n  - TRAILING PE: {xop.info.get("trailingPE")}\n  - FORWARD PE: {xop.info.get("forwardPE")}')

XLE: 
  - TRAILING PE: 20.7856
  - FORWARD PE: None
XOP: 
  - TRAILING PE: 13.45099
  - FORWARD PE: -16866.0


## 02. ASSET-DXY

* DX-Y.NYB: US Dollar Index

In [6]:
dxy = yf.Ticker('DX-Y.NYB').history(period='10y', interval='1d')[['Open', 'High', 'Low', 'Close', 'Volume']]
dxy.columns = ['open', 'high', 'low', 'close', 'volume']
dxy = dxy.tz_localize(None) if dxy.index.tz is not None else dxy

### 02.01 Correlation

In [7]:
import pylabwons as lw

rel = lw.DualRelation(dxy['close'], asset['XLE']['close'], indicator_name='DXY', asset_name='XLE')
rel.corr

KRX 로그인 실패: KRX_ID 또는 KRX_PW 환경 변수가 설정되지 않았습니다.


,leading,leading_diff(d),lagging,lagging_diff(d),synchronous(ccf),cumulative(raw),synchronous(raw),relation
2025/11-2026/05,-0.140068,14,-0.173799,10,0.095676,0.620885,0.047233,low correlation
2025/08-2026/02,0.279666,19,0.212252,9,-0.078473,0.677676,-0.362363,19d leading
2025/05-2025/11,0.184078,19,0.171295,10,0.155829,0.709561,-0.293714,low correlation
2025/02-2025/08,0.255009,4,-0.199560,4,0.230826,0.735549,0.396133,4d leading
2024/11-2025/05,0.238495,4,-0.185804,4,0.142152,0.757258,0.555550,low correlation
2024/08-2025/02,-0.173198,8,-0.157710,19,0.075086,0.753082,0.259565,low correlation
2024/05-2024/11,0.126013,10,-0.205256,18,0.153528,0.731451,0.412861,low correlation
2024/02-2024/08,0.165148,1,-0.150103,14,-0.112294,0.730562,0.433836,low correlation
2023/11-2024/05,0.149782,6,0.169092,10,-0.292156,0.712832,0.530841,synchronous
2023/08-2024/02,0.163639,11,-0.297567,19,-0.117362,0.693182,0.434545,19d lagging


### 02.02. Plot

In [8]:
import plotly.io as pio
pio.renderers.default = "vscode"

fig = rel.plotly()
fig.update_layout(height=600)

## 03. ASSET-FUTURES

* CL=F: WTI Crude Oil Futures
* BZ=F: Brent Crude Oil Futures
* NG=F: Natural Gas Futures

| 구분 | WTI (서부 텍사스산) | Brent (브렌트유) | Dubai (두바이유) |
| :--- | :--- | :--- | :--- |
| **TICKER** | CL=F | BZ=F | None |
| **기준 지역** | 미국 내륙 (텍사스 중심) | 유럽 북해 (해상) | 중동 (아랍에미리트) |
| **품질** | **최상** (가장 가벼운 경질유) | **우수** (고품질 경질유) | **보통** (황 함량 높은 중질유) |
| **운송 방식** | 파이프라인 (내륙 운송) | 유조선 (해상 운송 유리) | 유조선 (해상 운송 중심) |
| **주요 시장** | 미국 내 유가 지표 | **전 세계 유가 기준 (2/3)** | 아시아 시장 가격 기준 |
| **거래소** | NYMEX (뉴욕상업거래소) | ICE (런던선물거래소) | 현물 거래 중심 |


In [9]:
cl_f = yf.Ticker('CL=F')
bz_f = yf.Ticker('BZ=F')
ng_f = yf.Ticker('NG=F')
objs = {}
for ticker in [cl_f, bz_f, ng_f]:
    ohlcv = ticker.history(period='10y', interval='1d')[['Open', 'High', 'Low', 'Close', 'Volume']]
    ohlcv.columns = ['open', 'high', 'low', 'close', 'volume']
    objs[ticker.ticker] = ohlcv
futures = pd.concat(objs, axis=1)
futures = futures.tz_localize(None) if futures.index.tz is not None else futures

### 03.01. Correlation

In [10]:
sym = 'CL=F'
rel = lw.DualRelation(futures[sym]['close'], asset['XOP']['close'], indicator_name=sym, asset_name='XOP')
rel.corr

/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:393: RuntimeWarning:

invalid value encountered in log



,leading,leading_diff(d),lagging,lagging_diff(d),synchronous(ccf),cumulative(raw),synchronous(raw),relation
2025/11-2026/05,0.195791,20,0.253784,20,0.627152,0.578898,0.954035,synchronous
2025/08-2026/02,-0.246710,9,-0.178079,9,0.678805,0.549661,0.388747,synchronous
2025/05-2025/11,-0.219819,9,-0.186688,3,0.617500,0.560388,0.306682,synchronous
2025/02-2025/08,-0.258146,19,-0.147930,9,0.685084,0.566926,0.821818,synchronous
2024/11-2025/05,-0.256242,14,0.180740,20,0.714495,0.568273,0.826393,synchronous
2024/08-2025/02,-0.222178,7,-0.237574,18,0.538152,0.568110,0.381448,synchronous
2024/05-2024/11,0.163480,12,0.157392,1,0.584816,0.566342,0.826544,synchronous
2024/02-2024/08,0.191673,1,0.331594,9,0.496469,0.562962,0.699791,synchronous
2023/11-2024/05,-0.243143,8,0.145381,18,0.558902,0.551282,0.886400,synchronous
2023/08-2024/02,-0.194144,3,-0.176608,10,0.628349,0.534678,0.840830,synchronous


### 03.02. Plot

In [11]:
fig = rel.plotly()
fig.update_layout(height=600)

## 04. ASSET-OIL PRICE

* DCOILBRENTEU: Crude Oil Prices: Brent - Europe
* DCOILWTICO: Crude Oil Prices: West Texas Intermediate (WTI) - Cushing, Oklahoma 

In [12]:
today = datetime.now()
start = datetime(today.year - 10, today.month, today.day)
others = web.DataReader(['DCOILBRENTEU', 'DCOILWTICO'], 'fred', start, today)

### 04.01. Correlation

In [14]:
sym = 'DCOILWTICO'
rel = lw.DualRelation(others['DCOILWTICO'], asset['XLE']['close'], indicator_name=sym, asset_name='XLE')
rel.corr

/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:393: RuntimeWarning:

invalid value encountered in log



,leading,leading_diff(d),lagging,lagging_diff(d),synchronous(ccf),cumulative(raw),synchronous(raw),relation
2025/11-2026/05,0.265758,4,-0.170636,19,0.517365,0.646839,0.874122,synchronous
2025/08-2026/02,0.201932,17,0.139548,7,0.634150,0.632932,-0.148316,synchronous
2025/05-2025/11,0.207847,1,-0.157508,3,0.570640,0.672922,0.152632,synchronous
2025/02-2025/08,-0.265833,9,-0.172812,9,0.671493,0.700022,0.842919,synchronous
2024/11-2025/05,-0.321417,14,-0.158201,3,0.678817,0.716604,0.604727,synchronous
2024/07-2025/01,-0.175017,16,-0.196060,7,0.525712,0.735501,0.096267,synchronous
2024/04-2024/10,0.154402,12,-0.157138,5,0.598398,0.751760,0.508962,synchronous
2024/01-2024/07,0.184967,15,0.236615,9,0.472651,0.766323,0.722658,synchronous
2023/10-2024/04,-0.205853,8,0.178772,18,0.545949,0.768959,0.865676,synchronous
2023/07-2024/01,-0.211977,9,-0.199077,13,0.631315,0.769867,0.872636,synchronous


### 04.02. Plot

In [15]:
fig = rel.plotly()
fig.update_layout(height=600)

## 05. ASSET-PPI

* PCU213111213111: Producer Price Index by Industry: Drilling Oil and Gas Wells

In [16]:
today = datetime.now()
start = datetime(today.year - 10, today.month, today.day)
others = web.DataReader(['PCU213111213111'], 'fred', start, today)

### 05.01. Correlation

In [17]:
rel = lw.DualRelation(others['PCU213111213111'].dropna(), asset['XLE']['close'], asset_name='XLE', window=24, max_lag=6)
rel.corr

,leading,leading_diff(m),lagging,lagging_diff(m),synchronous(ccf),cumulative(raw),synchronous(raw),relation
2024/03-2026/03,0.393463,2,-0.250569,1,0.030782,0.930909,-0.462023,2m leading
2023/03-2025/03,0.411513,1,-0.289753,3,0.419991,0.939259,0.452758,synchronous
2022/03-2024/03,-0.559929,2,-0.323810,4,0.180465,0.918422,0.801696,2m leading
2021/03-2023/03,-0.327376,4,-0.416250,6,-0.030197,0.858749,0.950162,6m lagging
2020/03-2022/03,0.258619,3,0.232956,1,0.153167,0.580092,0.831943,3m leading
2019/03-2021/03,0.324156,1,0.274025,5,0.174528,0.513241,0.880947,1m leading
2018/03-2020/03,0.380227,6,0.464641,4,0.067987,0.000072,0.181003,4m lagging
2017/03-2019/03,0.216814,6,-0.302136,1,-0.086932,0.130482,0.035914,1m lagging


### 05.02. Plot

In [18]:
fig = rel.plotly()
fig.update_layout(height=600)

## 06. ASSET-TREASURY

In [19]:
today = datetime.now()
start = datetime(today.year - 10, today.month, today.day)
others = web.DataReader(['DGS10', 'T10Y2Y', 'BAMLH0A0HYM2'], 'fred', start, today)

### 06.01. Correlation

In [20]:
rel = lw.DualRelation(others['BAMLH0A0HYM2'].dropna(), asset['XLE']['close'], asset_name='XLE')
rel.corr

,leading,leading_diff(d),lagging,lagging_diff(d),synchronous(ccf),cumulative(raw),synchronous(raw),relation
2025/11-2026/05,-0.174953,14,-0.189828,14,0.149132,-0.520040,0.329055,low correlation
2025/08-2026/02,0.323962,11,-0.160360,12,-0.198652,-0.744223,-0.350647,11d leading
2025/05-2025/11,-0.179768,14,-0.233204,3,-0.330395,-0.757562,-0.792811,synchronous
2025/02-2025/08,0.254623,19,-0.216806,20,-0.577842,-0.745268,-0.678094,synchronous
2024/11-2025/05,0.256951,14,-0.220166,20,-0.583641,-0.756974,-0.709330,synchronous
2024/08-2025/02,-0.209408,1,0.274178,18,-0.327294,-0.756499,-0.570403,synchronous
2024/05-2024/11,-0.153916,8,-0.202225,1,-0.397051,-0.738573,-0.529175,synchronous
2024/02-2024/08,-0.253925,1,-0.173388,1,-0.329775,-0.715979,-0.484600,synchronous
2023/11-2024/05,-0.251323,18,0.200426,15,-0.232032,-0.625641,-0.697939,18d leading
2023/08-2024/02,-0.256952,18,0.192596,16,-0.230110,-0.327975,0.377347,18d leading


### 06.02. Plot

In [21]:
fig = rel.plotly()
fig.update_layout(height=600)